# Audio statistics — DCASE Task 2 dev / additional / eval

Audit waveform power and log-mel dB ranges across the three DCASE 2020 Task 2 dataset
roots used in this project so we can pick a principled clamp range for log-mel inputs.

For each dataset we compute, **per-file**:

- waveform peak / RMS (linear and dB)
- log-mel dB min / max (from `MelSpectrogram(power=2)` followed by `10·log10`, ref=1.0)

then we:

1. summarize the distribution across files (min, p1, p99, max),
2. compare the three datasets,
3. propose a `(CLAMP_MIN, CLAMP_MAX)` pair and quantify how much content it would clip,
4. break the dB ranges down by machine type.

The MelSpectrogram parameters match those used by `src.data.dataset.DCASE2020Task2LogMelDataset`
(`n_fft=1024`, `hop_length=512`, `n_mels=128`, `power=2.0`, `sr=16000`).

## 1. Imports and dataset roots

In [ ]:
%matplotlib inline

import sys
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torchaudio
from tqdm.auto import tqdm


def find_project_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(10):
        if (p / "src" / "data" / "dataset.py").is_file():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise FileNotFoundError("Could not find project root (src/data/dataset.py).")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


def resolve_dataset_root(name: str) -> Path:
    """Try a few known locations for a DCASE Task 2 dataset root."""
    candidates = [
        Path(f"/mnt/ssd/LaCie/dcase2020_task2/{name}"),
        Path(f"/mnt/ssd/LaCie/{name}"),
        PROJECT_ROOT / "dataset" / name,
        PROJECT_ROOT / "data" / name,
    ]
    for c in candidates:
        if c.exists():
            return c
    return candidates[0]  # return first candidate even if missing, for an informative error


DATASETS = {
    "dev": resolve_dataset_root("dcase2020_task2_dev_dataset"),
    "additional": resolve_dataset_root("dcase2020_task2_additional_train_dataset"),
    "eval": resolve_dataset_root("dcase2020_task2_eval_dataset"),
}

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
for tag, root in DATASETS.items():
    print(f"  {tag:11s} -> {root}  (exists={root.exists()})")

## 2. `audit_dataset_levels` — waveform & log-mel dB statistics

Scan every `.wav` under a dataset root and collect:

- waveform: peak amplitude, RMS, peak dB, RMS dB
- log-mel: per-file min and max dB after `10·log10(power)` with `power=2.0` and a small floor (`1e-10`)

No clamping is applied here — this is a dry run to see the natural dynamic range. Files are
mono-mixed (channel-averaged) and resampled to 16 kHz when needed so the stats line up with
how `DCASE2020Task2LogMelDataset` consumes the audio.

In [ ]:
SAMPLE_RATE = 16_000
N_FFT = 1024
HOP_LENGTH = 512
N_MELS = 128


def _make_mel() -> torchaudio.transforms.MelSpectrogram:
    return torchaudio.transforms.MelSpectrogram(
        sample_rate=SAMPLE_RATE,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        power=2.0,
        norm="slaney",
        mel_scale="htk",
    )


def _load_mono(path: Path) -> torch.Tensor:
    """Load a wav as mono float32 at SAMPLE_RATE (matches dataset.py preprocessing)."""
    wav, sr = torchaudio.load(str(path))
    if sr != SAMPLE_RATE:
        wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)
    if wav.shape[0] > 1:
        wav = wav.mean(0, keepdim=True)
    return wav


def audit_dataset_levels(
    dataset_root: str | Path,
    n_files: int | None = None,
    desc: str | None = None,
) -> dict[str, list[float]]:
    """
    Scan all WAV files and report power/dB statistics.
    Run this BEFORE deciding on any clamp range.
    """
    root = Path(dataset_root)
    paths = sorted(root.rglob("*.wav"))
    if n_files:
        paths = paths[:n_files]
    if not paths:
        raise FileNotFoundError(f"No .wav files under {root}")

    results: dict[str, list[float]] = {
        "peak_linear": [],
        "rms_linear": [],
        "peak_db": [],
        "rms_db": [],
        "mel_db_min": [],
        "mel_db_max": [],
    }

    mel_transform = _make_mel()

    for path in tqdm(paths, desc=desc or root.name, leave=False):
        waveform = _load_mono(path)

        peak = waveform.abs().max().item()
        rms = waveform.pow(2).mean().sqrt().item()
        results["peak_linear"].append(peak)
        results["rms_linear"].append(rms)
        results["peak_db"].append(20 * np.log10(peak + 1e-10))
        results["rms_db"].append(20 * np.log10(rms + 1e-10))

        power = mel_transform(waveform).clamp(min=1e-10)
        log_mel = 10.0 * torch.log10(power)
        results["mel_db_min"].append(log_mel.min().item())
        results["mel_db_max"].append(log_mel.max().item())

    print(f"\n{root.name}  ({len(paths)} files)")
    print("-" * 76)
    for key, vals in results.items():
        arr = np.asarray(vals)
        print(
            f"  {key:14s}  min={arr.min():9.3f}  "
            f"p1={np.percentile(arr, 1):9.3f}  "
            f"p99={np.percentile(arr, 99):9.3f}  "
            f"max={arr.max():9.3f}"
        )
    return results

## 3. Run the audit on each dataset

This pass scans every `.wav` under each root (no `n_files` cap). Set `N_FILES_LIMIT`
below to e.g. `200` for a quick smoke run.

In [ ]:
N_FILES_LIMIT: int | None = None  # set to e.g. 200 for a quick subset run

audits: dict[str, dict[str, list[float]]] = {}
for tag, root in DATASETS.items():
    if not root.exists():
        print(f"[skip] {tag}: {root} does not exist")
        continue
    audits[tag] = audit_dataset_levels(root, n_files=N_FILES_LIMIT, desc=tag)

print(f"\nAudited {len(audits)} dataset(s): {list(audits.keys())}")

## 4. Compare across datasets

Side-by-side summary table and histograms of the per-file log-mel dB extremes
(`mel_db_min`, `mel_db_max`). The histograms make tail behaviour easy to see —
this is what informs the clamp range below.

In [ ]:
def summary_row(name: str, vals: list[float]) -> tuple[str, ...]:
    arr = np.asarray(vals)
    return (
        name,
        f"{arr.min():9.3f}",
        f"{np.percentile(arr, 1):9.3f}",
        f"{np.percentile(arr, 50):9.3f}",
        f"{np.percentile(arr, 99):9.3f}",
        f"{arr.max():9.3f}",
    )


METRICS = ["peak_db", "rms_db", "mel_db_min", "mel_db_max"]

if audits:
    header = ("dataset / metric", "min", "p1", "p50", "p99", "max")
    print(f"{header[0]:32s} {header[1]:>9s} {header[2]:>9s} {header[3]:>9s} {header[4]:>9s} {header[5]:>9s}")
    print("-" * 80)
    for metric in METRICS:
        for tag, res in audits.items():
            row = summary_row(f"{tag:11s} | {metric:14s}", res[metric])
            print(f"{row[0]:32s} {row[1]:>9s} {row[2]:>9s} {row[3]:>9s} {row[4]:>9s} {row[5]:>9s}")
        print("-" * 80)

if audits:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=False)
    colors = plt.cm.tab10.colors

    for ax, key, title in zip(
        axes,
        ("mel_db_min", "mel_db_max"),
        ("Per-file log-mel dB minimum", "Per-file log-mel dB maximum"),
    ):
        for i, (tag, res) in enumerate(audits.items()):
            arr = np.asarray(res[key])
            ax.hist(
                arr,
                bins=60,
                alpha=0.5,
                label=f"{tag} (n={len(arr)})",
                color=colors[i % len(colors)],
            )
            ax.axvline(np.percentile(arr, 1), color=colors[i % len(colors)], lw=1, ls=":")
            ax.axvline(np.percentile(arr, 99), color=colors[i % len(colors)], lw=1, ls=":")
        ax.set_xlabel("dB")
        ax.set_ylabel("# files")
        ax.set_title(title)
        ax.grid(True, alpha=0.3)
        ax.legend()

    plt.tight_layout()
    plt.show()

## 5. `report_clamp_loss` — choose a clamp range

Pick `(CLAMP_MIN, CLAMP_MAX)` with headroom past the observed p1 / p99 and check how many
files have **any** content outside the bounds across all three roots combined.

`src.utils.audio.mel_db_to_finite` currently uses a fixed `top_db=80.0` clamp (i.e. clamp to
`[-80, +∞)` after `AmplitudeToDB(top_db=80)` already trims the floor 80 dB below per-clip
peak). The values here let you sanity-check whether a uniform absolute clamp works for all
three datasets, or whether per-clip `AmplitudeToDB(top_db=...)` is still the better choice.

In [ ]:
def report_clamp_loss(results: dict[str, list[float]], clamp_min: float, clamp_max: float) -> tuple[float, float]:
    mins = np.asarray(results["mel_db_min"])
    maxs = np.asarray(results["mel_db_max"])
    pct_floor_clipped = (mins < clamp_min).mean() * 100
    pct_ceil_clipped = (maxs > clamp_max).mean() * 100
    print(f"  files with content below {clamp_min:+.1f} dB: {pct_floor_clipped:5.2f}%")
    print(f"  files with content above {clamp_max:+.1f} dB: {pct_ceil_clipped:5.2f}%")
    return pct_floor_clipped, pct_ceil_clipped


# Suggest bounds: round p1 of the per-file mel_db_min DOWN and p99 of mel_db_max UP, with headroom.
if audits:
    all_mel_min = np.concatenate([np.asarray(r["mel_db_min"]) for r in audits.values()])
    all_mel_max = np.concatenate([np.asarray(r["mel_db_max"]) for r in audits.values()])

    p1_min = np.percentile(all_mel_min, 1)
    p99_max = np.percentile(all_mel_max, 99)

    HEADROOM_DB = 5.0
    CLAMP_MIN = float(np.floor((p1_min - HEADROOM_DB) / 5.0) * 5.0)
    CLAMP_MAX = float(np.ceil((p99_max + HEADROOM_DB) / 5.0) * 5.0)

    print("Pooled across all audited datasets:")
    print(f"  mel_db_min   p1 = {p1_min:+.2f}   abs min = {all_mel_min.min():+.2f}")
    print(f"  mel_db_max  p99 = {p99_max:+.2f}   abs max = {all_mel_max.max():+.2f}")
    print(f"\nSuggested clamp:  CLAMP_MIN = {CLAMP_MIN:+.1f}   CLAMP_MAX = {CLAMP_MAX:+.1f}")
    print(f"(p1/p99 with ±{HEADROOM_DB:.0f} dB headroom, snapped to 5 dB)\n")

    print("Per-dataset clamp loss at suggested bounds:")
    for tag, res in audits.items():
        print(f"\n[{tag}]")
        report_clamp_loss(res, CLAMP_MIN, CLAMP_MAX)
else:
    CLAMP_MIN, CLAMP_MAX = -100.0, 10.0
    print("No audits available; falling back to defaults CLAMP_MIN/MAX.")

## 6. `audit_by_machine_type`

Same log-mel dB extremes, but bucketed by machine type (`fan`, `pump`, `slider`, `valve`,
`ToyCar`, `ToyConveyor`, ...). The path layout is `{root}/{machine_type}/{train|test}/*.wav`,
so `path.parts[-3]` is the machine type.

In [ ]:
def audit_by_machine_type(
    dataset_root: str | Path,
    n_files_per_type: int | None = None,
    desc: str | None = None,
) -> dict[str, list[tuple[float, float]]]:
    """Group per-file (mel_db_min, mel_db_max) by machine type."""
    root = Path(dataset_root)
    paths = sorted(root.rglob("*.wav"))
    if not paths:
        raise FileNotFoundError(f"No .wav files under {root}")

    mel_transform = _make_mel()

    if n_files_per_type is not None:
        per_type_paths: dict[str, list[Path]] = defaultdict(list)
        for p in paths:
            mt = p.parts[-3]
            if len(per_type_paths[mt]) < n_files_per_type:
                per_type_paths[mt].append(p)
        paths = [p for ps in per_type_paths.values() for p in ps]

    per_type: dict[str, list[tuple[float, float]]] = defaultdict(list)
    for path in tqdm(paths, desc=desc or root.name, leave=False):
        machine_type = path.parts[-3]
        waveform = _load_mono(path)
        power = mel_transform(waveform).clamp(min=1e-10)
        log_mel = 10.0 * torch.log10(power)
        per_type[machine_type].append((log_mel.min().item(), log_mel.max().item()))

    print(f"\n{root.name}  (by machine type)")
    print("-" * 80)
    print(f"{'machine_type':15s}  {'n':>5s}  {'db_min [min .. p99]':>26s}  {'db_max [p1 .. max]':>26s}")
    print("-" * 80)
    for mtype, vals in sorted(per_type.items()):
        mins = np.asarray([v[0] for v in vals])
        maxs = np.asarray([v[1] for v in vals])
        print(
            f"{mtype:15s}  {len(vals):5d}  "
            f"[{mins.min():+7.1f} .. {np.percentile(mins, 99):+7.1f}]  "
            f"[{np.percentile(maxs, 1):+7.1f} .. {maxs.max():+7.1f}]"
        )
    return per_type

In [ ]:
N_FILES_PER_TYPE_LIMIT: int | None = None  # set to e.g. 50 for a quick subset run

per_type_audits: dict[str, dict[str, list[tuple[float, float]]]] = {}
for tag, root in DATASETS.items():
    if not root.exists():
        print(f"[skip] {tag}: {root} does not exist")
        continue
    per_type_audits[tag] = audit_by_machine_type(
        root,
        n_files_per_type=N_FILES_PER_TYPE_LIMIT,
        desc=tag,
    )

In [ ]:
if per_type_audits:
    all_types = sorted({mt for d in per_type_audits.values() for mt in d.keys()})
    n_types = len(all_types)
    n_ds = len(per_type_audits)

    fig, axes = plt.subplots(2, 1, figsize=(max(8, 0.6 * n_types * n_ds + 4), 7), sharex=True)

    width = 0.8 / max(n_ds, 1)
    x = np.arange(n_types, dtype=float)
    colors = plt.cm.tab10.colors

    for i, (tag, per_type) in enumerate(per_type_audits.items()):
        mins_pos: list[float] = []
        mins_lo: list[float] = []
        mins_hi: list[float] = []
        maxs_pos: list[float] = []
        maxs_lo: list[float] = []
        maxs_hi: list[float] = []
        for j, mt in enumerate(all_types):
            if mt not in per_type:
                continue
            vals = per_type[mt]
            mins = np.asarray([v[0] for v in vals])
            maxs = np.asarray([v[1] for v in vals])
            mins_pos.append(x[j] + (i - (n_ds - 1) / 2) * width)
            mins_lo.append(mins.min())
            mins_hi.append(np.percentile(mins, 99))
            maxs_pos.append(x[j] + (i - (n_ds - 1) / 2) * width)
            maxs_lo.append(np.percentile(maxs, 1))
            maxs_hi.append(maxs.max())

        if mins_pos:
            axes[0].vlines(
                mins_pos,
                mins_lo,
                mins_hi,
                lw=4,
                color=colors[i % len(colors)],
                label=tag,
                alpha=0.85,
            )
        if maxs_pos:
            axes[1].vlines(
                maxs_pos,
                maxs_lo,
                maxs_hi,
                lw=4,
                color=colors[i % len(colors)],
                label=tag,
                alpha=0.85,
            )

    axes[0].axhline(CLAMP_MIN, color="k", ls="--", lw=1, label=f"CLAMP_MIN={CLAMP_MIN:+.1f}")
    axes[1].axhline(CLAMP_MAX, color="k", ls="--", lw=1, label=f"CLAMP_MAX={CLAMP_MAX:+.1f}")

    axes[0].set_ylabel("mel_db_min  [min .. p99]")
    axes[0].set_title("Per-file log-mel dB minimum, by machine type")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(ncol=min(4, n_ds + 1))

    axes[1].set_ylabel("mel_db_max  [p1 .. max]")
    axes[1].set_title("Per-file log-mel dB maximum, by machine type")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(all_types, rotation=30, ha="right")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend(ncol=min(4, n_ds + 1))

    plt.tight_layout()
    plt.show()